In [2]:
import anthropic
import duckdb
import os
import json
import yaml
from pathlib import Path
from dotenv import load_dotenv

In [3]:
# Load API key from .env file
load_dotenv()

True

In [4]:
# Setup
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
conn = duckdb.connect(str(Path("olist_pipeline/dev.duckdb")))

In [5]:
PROJECT_DIR = Path("olist_pipeline")
MODELS_DIR = PROJECT_DIR / "models"
MODEL_LAYERS = ["staging", "intermediate", "marts"]

In [6]:
print("Setup complete")
print(f"Claude client ready: {client is not None}")
print(f"DuckDB connected: {conn is not None}")
print(f"Models directory exists: {MODELS_DIR.exists()}")

Setup complete
Claude client ready: True
DuckDB connected: True
Models directory exists: True


In [7]:
def get_model_sql(sql_path: Path) -> str:
    """Read SQL file content"""
    return sql_path.read_text(encoding="utf-8")

In [8]:
def get_model_columns(model_name: str) -> list:
    """Query DuckDB to get actual column names and types"""
    try:
        result = conn.execute(f"describe {model_name}").fetchall()
        return [{"name": row[0], "type": row[1]} for row in result]
    except Exception as e:
        print(f"Could not describe {model_name}: {e}")
        return []

In [9]:
def get_sample_data(model_name: str, limit: int = 3) -> str:
    """Get sample rows to help Claude understand the data"""
    try:
        result = conn.execute(f"select * from {model_name} limit {limit}").fetchdf()
        return result.to_string(index=False)
    except Exception:
        return "No sample data available"

In [10]:
test_model = "stg_orders"
columns = get_model_columns(test_model)
print(f"Columns in {test_model}:")
for col in columns:
    print(f"  {col['name']} — {col['type']}")

Columns in stg_orders:
  order_id — VARCHAR
  customer_id — VARCHAR
  order_status — VARCHAR
  ordered_at — TIMESTAMP
  approved_at — TIMESTAMP
  shipped_at — TIMESTAMP
  delivered_at — TIMESTAMP
  estimated_delivery_at — TIMESTAMP


In [11]:
def count_tokens(prompt: str) -> int:
    """Count exact input tokens before API call"""
    response = client.messages.count_tokens(
        model="claude-sonnet-4-6",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.input_tokens

In [12]:
test_prompt = "What is dbt?"
tokens = count_tokens(test_prompt)
print(f"Test prompt: '{test_prompt}'")
print(f"Token count: {tokens}")

Test prompt: 'What is dbt?'
Token count: 12


In [29]:
def generate_model_documentation(model_name: str, sql: str, columns: list, sample: str) -> dict:
    """Send model info to Claude and get plain text documentation back"""
    
    columns_text = "\n".join([f"- {col['name']} ({col['type']})" for col in columns])
    
    prompt = f"""You are a senior analytics engineer writing dbt documentation.

MODEL NAME: {model_name}

SQL CODE:
{sql}

SAMPLE DATA:
{sample}

Write documentation for this model and each of these columns:
{columns_text}

Return in this exact format, one line per item:
MODEL: your model description here
{chr(10).join([f"{col['name']}: your description here" for col in columns])}

Rules:
- MODEL line: maximum 20 words
- Each column line: maximum 15 words
- Replace "your description here" with the actual description
- No extra lines, no JSON, no markdown
"""

    # Count input tokens exactly
    input_tokens = count_tokens(prompt)
    
    # Output budget based on constraints
    # 20 words model + 15 words per column, all x 1.3 tokens/word + 40% buffer
    estimated_tokens = 26 + (len(columns) * 20) + 50
    buffer = max(100, len(columns) * 10)
    output_tokens = estimated_tokens + buffer

    print(f"  Input tokens (exact):      {input_tokens}")
    print(f"  Output tokens (estimated): {output_tokens}")
    print(f"  Estimated cost:            ${((input_tokens * 3) + (output_tokens * 15)) / 1_000_000:.6f}")

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=output_tokens,
        messages=[{"role": "user", "content": prompt}]
    )

    actual_input = response.usage.input_tokens
    actual_output = response.usage.output_tokens
    actual_cost = ((actual_input * 3) + (actual_output * 15)) / 1_000_000

    print(f"  Actual input tokens:       {actual_input}")
    print(f"  Actual output tokens:      {actual_output}")
    print(f"  Actual cost:               ${actual_cost:.6f}")

    # Parse plain text response — Claude returns simple lines, Python builds the structure
    text = response.content[0].text.strip()
    lines = text.split('\n')

    model_description = ""
    column_docs = []
    column_names = {col['name'] for col in columns}

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith("MODEL:"):
            model_description = line.replace("MODEL:", "").strip()
        elif ":" in line:
            parts = line.split(":", 1)
            col_name = parts[0].strip()
            col_desc = parts[1].strip()
            if col_name in column_names:
                column_docs.append({
                    "name": col_name,
                    "description": col_desc
                })

    doc = {
        "model_description": model_description,
        "columns": column_docs
    }

    return doc, actual_input, actual_output

print("Function defined successfully")

Function defined successfully


In [30]:
# Test on stg_orders
model_name = "stg_orders"
sql_path = MODELS_DIR / "staging" / f"{model_name}.sql"

print(f"Generating documentation for: {model_name}")
print("-" * 50)

sql = get_model_sql(sql_path)
columns = get_model_columns(model_name)
sample = get_sample_data(model_name)

doc, input_tokens, output_tokens = generate_model_documentation(
    model_name, sql, columns, sample
)

print("\n--- GENERATED DOCUMENTATION ---\n")
print(f"Model: {model_name}")
print(f"Description: {doc['model_description']}")
print(f"\nColumns ({len(doc['columns'])}):")
for col in doc['columns']:
    print(f"  {col['name']}: {col['description']}")

Generating documentation for: stg_orders
--------------------------------------------------
  Input tokens (exact):      741
  Output tokens (estimated): 336
  Estimated cost:            $0.007263
  Actual input tokens:       741
  Actual output tokens:      154
  Actual cost:               $0.004533

--- GENERATED DOCUMENTATION ---

Model: stg_orders
Description: Staged order data from Olist, capturing the full lifecycle of each customer order.

Columns (8):
  order_id: Unique identifier for each order placed on the platform.
  customer_id: Unique identifier for the customer who placed the order.
  order_status: Current status of the order, e.g. delivered, shipped, cancelled.
  ordered_at: Timestamp when the customer placed the order.
  approved_at: Timestamp when the order payment was approved.
  shipped_at: Timestamp when the order was handed to the carrier.
  delivered_at: Timestamp when the order was delivered to the customer.
  estimated_delivery_at: Estimated timestamp for the o

In [ ]:
def write_to_yaml(model_name: str, doc: dict, layer: str):
    """Update existing YAML file with generated descriptions, preserving tests.
    If model doesn't exist in YAML, adds it automatically."""
    
    yaml_files = {
        "staging": MODELS_DIR / "staging" / "staging.yml",
        "intermediate": MODELS_DIR / "intermediate" / "intermediate.yml",
        "marts": MODELS_DIR / "marts" / "marts.yml"
    }
    
    yaml_path = yaml_files[layer]
    
    # Read existing YAML
    with open(yaml_path, "r", encoding="utf-8") as f:
        existing = yaml.safe_load(f)

    # Check if model already exists
    model_names = [m["name"] for m in existing["models"]]
    
    if model_name in model_names:
        # Update existing model entry
        for model in existing["models"]:
            if model["name"] == model_name:
                model["description"] = doc["model_description"]
                
                generated_cols = {col["name"]: col["description"] for col in doc["columns"]}
                
                if "columns" in model:
                    for col in model["columns"]:
                        if col["name"] in generated_cols:
                            col["description"] = generated_cols[col["name"]]
                else:
                    model["columns"] = [
                        {"name": col["name"], "description": col["description"]}
                        for col in doc["columns"]
                    ]
                break
        print(f"  Updated existing entry in: {yaml_path.name}")
    
    else:
        # Model doesn't exist — add it automatically
        new_model = {
            "name": model_name,
            "description": doc["model_description"],
            "columns": [
                {"name": col["name"], "description": col["description"]}
                for col in doc["columns"]
            ]
        }
        existing["models"].append(new_model)
        print(f"  Added new entry to: {yaml_path.name}")
    
    # Write back to same file
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(existing, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print("Function updated successfully")

Function updated successfully


In [35]:
total_input_tokens = 0
total_output_tokens = 0
processed = []
failed = []

for layer in MODEL_LAYERS:
    layer_dir = MODELS_DIR / layer
    sql_files = list(layer_dir.glob("*.sql"))
    
    print(f"\n{'='*50}")
    print(f"Layer: {layer} ({len(sql_files)} models)")
    print(f"{'='*50}")
    
    for sql_path in sql_files:
        model_name = sql_path.stem
        print(f"\nProcessing: {model_name}")
        
        sql = get_model_sql(sql_path)
        columns = get_model_columns(model_name)
        sample = get_sample_data(model_name)
        
        if not columns:
            print(f"  Skipping — not found in database")
            failed.append(model_name)
            continue
        
        try:
            doc, input_tokens, output_tokens = generate_model_documentation(
                model_name, sql, columns, sample
            )
            write_to_yaml(model_name, doc, layer)
            
            total_input_tokens += input_tokens
            total_output_tokens += output_tokens
            processed.append(model_name)
            print(f"  ✓ Done")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
            failed.append(model_name)

# Final summary
total_cost = ((total_input_tokens * 3) + (total_output_tokens * 15)) / 1_000_000

print(f"\n{'='*50}")
print(f"COMPLETE")
print(f"{'='*50}")
print(f"Processed: {len(processed)} models")
print(f"Failed:    {len(failed)} models")
print(f"Total input tokens:  {total_input_tokens}")
print(f"Total output tokens: {total_output_tokens}")
print(f"Total cost:          ${total_cost:.6f}")
if failed:
    print(f"Failed models: {failed}")


Layer: staging (9 models)

Processing: stg_customers
  Input tokens (exact):      447
  Output tokens (estimated): 276
  Estimated cost:            $0.005481
  Actual input tokens:       447
  Actual output tokens:      110
  Actual cost:               $0.002991
  Updated: staging.yml
  ✓ Done

Processing: stg_geolocation
  Input tokens (exact):      362
  Output tokens (estimated): 276
  Estimated cost:            $0.005226
  Actual input tokens:       362
  Actual output tokens:      92
  Actual cost:               $0.002466
  Updated: staging.yml
  ✓ Done

Processing: stg_orders
  Input tokens (exact):      741
  Output tokens (estimated): 336
  Estimated cost:            $0.007263
  Actual input tokens:       741
  Actual output tokens:      156
  Actual cost:               $0.004563
  Updated: staging.yml
  ✓ Done

Processing: stg_order_items
  Input tokens (exact):      626
  Output tokens (estimated): 316
  Estimated cost:            $0.006618
  Actual input tokens:       626
 

In [ ]:
# Check one generated file
sample_file = MODELS_DIR / "marts" / "staging.yml"
print(sample_file.read_text())

version: 2
models:
- name: stg_orders
  description: Staged orders data from Olist, capturing order lifecycle timestamps
    and current status.
  columns:
  - name: order_id
    description: Unique identifier for each customer order placed on Olist.
    tests:
    - unique
    - not_null
  - name: customer_id
    description: Unique identifier linking the order to a specific customer.
    tests:
    - not_null
  - name: order_status
    description: Current status of the order (e.g., delivered, shipped, canceled).
    tests:
    - not_null
    - accepted_values:
        arguments:
          values:
          - delivered
          - shipped
          - canceled
          - unavailable
          - invoiced
          - processing
          - created
          - approved
  - name: ordered_at
    description: Timestamp when the customer placed the order.
  - name: approved_at
    description: Timestamp when the order payment was approved.
  - name: shipped_at
    description: Timestamp whe